### Phase 4: Importing and pulling weather report details for analysis
Input: Precipitation and Temperature readings from NOAA of 2025

In [1]:
!pip install noaa-sdk pandas

In [2]:
from google.colab import auth
auth.authenticate_user()

In [3]:
from google.cloud import bigquery
client = bigquery.Client(project="big-data-algorithms-493312")

In [4]:
# Import the data tables
!wget https://www.ncei.noaa.gov/pub/data/cirs/climdiv/climdiv-tmaxst-v1.0.0-20260406
!wget https://www.ncei.noaa.gov/pub/data/cirs/climdiv/climdiv-tminst-v1.0.0-20260406
!wget https://www.ncei.noaa.gov/pub/data/cirs/climdiv/climdiv-pcpnst-v1.0.0-20260406

--2026-04-21 13:42:12--  https://www.ncei.noaa.gov/pub/data/cirs/climdiv/climdiv-tmaxst-v1.0.0-20260406
Resolving www.ncei.noaa.gov (www.ncei.noaa.gov)... 205.167.25.168, 205.167.25.178, 205.167.25.171, ...
Connecting to www.ncei.noaa.gov (www.ncei.noaa.gov)|205.167.25.168|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1255380 (1.2M)
Saving to: ‘climdiv-tmaxst-v1.0.0-20260406’

climdiv-tmaxst-v1.0 100%[===================>]   1.20M  6.94MB/s    in 0.2s    

2026-04-21 13:42:13 (6.94 MB/s) - ‘climdiv-tmaxst-v1.0.0-20260406’ saved [1255380/1255380]

--2026-04-21 13:42:13--  https://www.ncei.noaa.gov/pub/data/cirs/climdiv/climdiv-tminst-v1.0.0-20260406
Resolving www.ncei.noaa.gov (www.ncei.noaa.gov)... 205.167.25.168, 205.167.25.178, 205.167.25.171, ...
Connecting to www.ncei.noaa.gov (www.ncei.noaa.gov)|205.167.25.168|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1255380 (1.2M)
Saving to: ‘climdiv-tminst-v1.0.0-20260406’

climdiv-tmi

In [5]:
import pandas as pd

state_code_map = {
    "001": "AL", "002": "AZ", "003": "AR", "004": "CA", "005": "CO",
    "006": "CT", "007": "DE", "008": "FL", "009": "GA", "010": "ID",
    "011": "IL", "012": "IN", "013": "IA", "014": "KS", "015": "KY",
    "016": "LA", "017": "ME", "018": "MD", "019": "MA", "020": "MI",
    "021": "MN", "022": "MS", "023": "MO", "024": "MT", "025": "NE",
    "026": "NV", "027": "NH", "028": "NJ", "029": "NM", "030": "NY",
    "031": "NC", "032": "ND", "033": "OH", "034": "OK", "035": "OR",
    "036": "PA", "037": "RI", "038": "SC", "039": "SD", "040": "TN",
    "041": "TX", "042": "UT", "043": "VT", "044": "VA", "045": "WA",
    "046": "WV", "047": "WI", "048": "WY", "050": "AK"
}

def parse_statewide_file(filename, value_name):
    rows = []

    with open(filename, "r") as f:
        for line in f:
            state_code = line[0:3]
            division_flag = line[3]  # should be '0' for statewide
            year = int(line[6:10])

            if division_flag != "0":
                continue

            if state_code not in state_code_map:
                continue

            if year != 2025:
                continue

            state = state_code_map[state_code]

            for i in range(12):
                start = 10 + i * 7
                raw = line[start:start+7].strip()

                if raw in ["-99.90", "-9.99", "-9999."]:
                    value = None
                else:
                    value = float(raw)

                rows.append({
                    "state": state,
                    "year_month": int(f"{year}{i+1:02d}"),
                    value_name: value
                })

    return pd.DataFrame(rows)

The above step imports pandas, but also converts the weather data formatting into something that is easier to understand. The NOAA files identify states using numeric area codes rather than two-letter abbreviations. By changing it, we can specifically join the weather data to the spend data later.

Since the NOAA files are not CSVs, we need to convert them into a more reasonable format for our research purposes. Each line currently. The function reads each line, keeps only the statewide rows for 2025 (since the data from Aramark is only from 2025), extracts the 12 monthly values, and turns them into a tidy dataframe stores metadata plus 12 monthly values in a fixed-width format.

In [6]:
tmax = parse_statewide_file("climdiv-tmaxst-v1.0.0-20260406", "tmax_f")
tmin = parse_statewide_file("climdiv-tminst-v1.0.0-20260406", "tmin_f")
precip = parse_statewide_file("climdiv-pcpnst-v1.0.0-20260406", "total_precip_inches")

#Computer average temperature, rather than having max and min.
temp = tmax.merge(tmin, on=["state", "year_month"], how="inner")
temp["avg_temp_f"] = (temp["tmax_f"] + temp["tmin_f"]) / 2
temp = temp[["state", "year_month", "avg_temp_f"]]

# Merge into one Weather table, for easy access
weather = temp.merge(precip, on=["state", "year_month"], how="inner")
weather = weather[["state", "year_month", "avg_temp_f", "total_precip_inches"]]
weather.head()

,state,year_month,avg_temp_f,total_precip_inches
0,AL,202501,40.10,3.24
1,AL,202502,52.70,4.45
2,AL,202503,58.25,5.20
3,AL,202504,66.80,5.65
4,AL,202505,72.00,9.83


Because we downloaded maximum and minimum temperature rather than direct average temperature, this step estimates average temperature. On top of that, we merge the precipitation values into one table. This makes it more useful and reasonable to work with in future phases.

In [7]:
warm_months = weather[weather["avg_temp_f"] > 50]

rain_threshold = (
    warm_months
    .groupby("state")["total_precip_inches"]
    .quantile(0.25)
)

rain_threshold = rain_threshold.clip(lower=0.2)

weather["rain_threshold"] = weather["state"].map(rain_threshold)
weather["snow_proxy"] = (
    (weather["avg_temp_f"] <= 32) &
    (
        (weather["total_precip_inches"] >= weather["rain_threshold"]) |
        (weather["total_precip_inches"] > 0.1)
    )
).astype(int)

print(weather.groupby("snow_proxy")["avg_temp_f"].mean())
print(weather["snow_proxy"].value_counts())


snow_proxy
0    58.423360
1    22.613529
Name: avg_temp_f, dtype: float64
snow_proxy
0    503
1     85
Name: count, dtype: int64


Now it is not possible to get a specific measure of snowfall from the states, as most data summaries merge snowfall with precipitation. Yet snow storms, are a big reason why distribution can be impacted, which can impact the sales that occur. To account for this, a temporary snow estimation setup has been created.

Assuming the conditions for snowfall are temperature below or at freezing, we can parse through the average precipitation during the warm months. This sets a threshhold which can predict additional/replacement snowfall on top of this. This threshold also serves to point out abnormal periods of rain/storms that may occur over time.

The computed rain thresholds are mapped back to the full dataset so that each row is evaluated relative to its own state’s climate conditions rather than using a single global threshold. Eventually a prediction of snow can be accumulated, which is then merged with the weather table for further analysis of the situation.

In [8]:
final_weather = weather[
    ["state", "year_month", "avg_temp_f", "total_precip_inches", "rain_threshold", "snow_proxy"]
].copy()

print(final_weather.head())
print(final_weather.dtypes)
print("Rows:", len(final_weather))
print("States:", final_weather["state"].nunique())
print("Months:", final_weather["year_month"].nunique())

  state  year_month  avg_temp_f  total_precip_inches  rain_threshold  \
0    AL      202501       40.10                 3.24            3.01   
1    AL      202502       52.70                 4.45            3.01   
2    AL      202503       58.25                 5.20            3.01   
3    AL      202504       66.80                 5.65            3.01   
4    AL      202505       72.00                 9.83            3.01   

   snow_proxy  
0           0  
1           0  
2           0  
3           0  
4           0  
state                   object
year_month               int64
avg_temp_f             float64
total_precip_inches    float64
rain_threshold         float64
snow_proxy               int64
dtype: object
Rows: 588
States: 49
Months: 12


In [9]:
from google.cloud import bigquery

table_id = "big-data-algorithms-493312.aramark_spend.weather_state_monthly"

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

load_job = client.load_table_from_dataframe(
    final_weather,
    table_id,
    job_config=job_config
)

load_job.result()

print(f"Upload complete: {table_id}")

Upload complete: big-data-algorithms-493312.aramark_spend.weather_state_monthly


---

## Phase 4 Extension — Heating and Cooling Degree Days

The temperature and precipitation we already loaded cover the basics, but they don't directly quantify how *intense* a cold or hot month actually was. For that, we bring in two more variables from the same NOAA climdiv dataset:

- **Heating Degree Days (HDD)** — the sum of `max(0, 65°F − daily_avg_temp)` across every day in the month. Higher HDD = colder month that requires more heating. A standard metric used by utilities and chain operators to quantify cold intensity.
- **Cooling Degree Days (CDD)** — the mirror: sum of `max(0, daily_avg_temp − 65°F)`. Higher CDD = hotter month.

These are the industry-standard climate variables for demand analysis. The snow proxy we built earlier stays as a secondary binary flag, but HDD becomes our primary cold-intensity signal for Phase 5 correlations.

### Step 1 — Download the HDD and CDD files from NOAA

NOAA's CIRS (Climatological Indices Reference Series) bundles several statewide monthly variables as fixed-width text files served over plain HTTPS.

The date suffix in the filename is NOAA's internal release version, not the data date.

`wget` saves the files directly into the Colab filesystem alongside the temperature and precipitation files we already have.


In [10]:
# HDD and CDD statewide monthly files from the same NOAA CIRS climdiv dataset.
!wget https://www.ncei.noaa.gov/pub/data/cirs/climdiv/climdiv-hddcst-v1.0.0-20260406
!wget https://www.ncei.noaa.gov/pub/data/cirs/climdiv/climdiv-cddcst-v1.0.0-20260406


--2026-04-21 13:44:12--  https://www.ncei.noaa.gov/pub/data/cirs/climdiv/climdiv-hddcst-v1.0.0-20260406
Resolving www.ncei.noaa.gov (www.ncei.noaa.gov)... 205.167.25.171, 205.167.25.178, 205.167.25.172, ...
Connecting to www.ncei.noaa.gov (www.ncei.noaa.gov)|205.167.25.171|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 812028 (793K)
Saving to: ‘climdiv-hddcst-v1.0.0-20260406’

climdiv-hddcst-v1.0 100%[===================>] 793.00K  --.-KB/s    in 0.1s    

2026-04-21 13:44:13 (5.56 MB/s) - ‘climdiv-hddcst-v1.0.0-20260406’ saved [812028/812028]

--2026-04-21 13:44:13--  https://www.ncei.noaa.gov/pub/data/cirs/climdiv/climdiv-cddcst-v1.0.0-20260406
Resolving www.ncei.noaa.gov (www.ncei.noaa.gov)... 205.167.25.171, 205.167.25.178, 205.167.25.172, ...
Connecting to www.ncei.noaa.gov (www.ncei.noaa.gov)|205.167.25.171|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 812028 (793K)
Saving to: ‘climdiv-cddcst-v1.0.0-20260406’

climdiv-cddcst-

### Step 2 — Parse the HDD and CDD Files and Merge Into the Weather DataFrame

Because the HDD and CDD files share the exact same fixed-width layout as the temperature and precipitation files, we can reuse the `parse_statewide_file` function defined earlier without modification. The function walks each line of the file, extracts the state code, filters to 2025 statewide-level rows, and expands the 12 monthly values into a tidy long-format DataFrame.

Once we have the HDD and CDD DataFrames in memory, we merge them into the existing `final_weather` table on `(state, year_month)`. The `how="left"` join preserves every row of the original weather table — if for any reason a particular state-month row does not have HDD or CDD coverage, it comes through as `NaN`.


In [11]:
# Reuse the existing parse_statewide_file function. Same fixed-width layout.
hdd = parse_statewide_file("climdiv-hddcst-v1.0.0-20260406", "heating_degree_days")
cdd = parse_statewide_file("climdiv-cddcst-v1.0.0-20260406", "cooling_degree_days")

# Merge into the existing weather dataframe on (state, year_month).
weather_enriched = (
    final_weather
    .merge(hdd, on=["state", "year_month"], how="left")
    .merge(cdd, on=["state", "year_month"], how="left")
)

# Reorder columns for readability.
weather_enriched = weather_enriched[[
    "state",
    "year_month",
    "avg_temp_f",
    "total_precip_inches",
    "heating_degree_days",
    "cooling_degree_days",
    "rain_threshold",
    "snow_proxy",
]]

weather_enriched.head()


,state,year_month,avg_temp_f,total_precip_inches,heating_degree_days,cooling_degree_days,rain_threshold,snow_proxy
0,AL,202501,40.10,3.24,802.0,2.0,3.01,0
1,AL,202502,52.70,4.45,400.0,14.0,3.01,0
2,AL,202503,58.25,5.20,269.0,38.0,3.01,0
3,AL,202504,66.80,5.65,63.0,98.0,3.01,0
4,AL,202505,72.00,9.83,20.0,209.0,3.01,0


### Step 3 — Sanity-check the Enriched DataFrame

Before we push the new table to BigQuery, we verify that the HDD and CDD values behave the way the physics says they should.

Heating degree days are the sum of `max(0, 65°F − daily_mean_temp)` over every day of the month. A cold northern state like Minnesota should have HDD values well above 1,000 in January and February, and effectively zero in July and August. Cooling degree days are the mirror: `max(0, daily_mean_temp − 65°F)` summed across the month, so a warm southern state like Florida should show year-round CDD and near-zero HDD.

Printing side-by-side tables for Minnesota and Florida surfaces any parsing mistake immediately — if the numbers look unreasonable (all zeros, wild outliers, or constant across months), something went wrong in the parse step. We also print descriptive statistics across all 588 rows to confirm that the overall ranges are plausible, and a null-count to make sure the merge did not leave any gaps.


In [12]:
# Quick sanity check: HDD should be high in winter and near-0 in summer; CDD the opposite.
# Pull HDD/CDD for one northern state (MN) and one southern state (FL) across all 12 months.
for s in ["MN", "FL"]:
    subset = (
        weather_enriched[weather_enriched["state"] == s]
        .sort_values("year_month")
        [["year_month", "avg_temp_f", "heating_degree_days", "cooling_degree_days"]]
    )
    print(f"\n--- {s} ---")
    print(subset.to_string(index=False))

# Summary stats
print("\n--- Overall summary ---")
print(weather_enriched[["heating_degree_days", "cooling_degree_days"]].describe().round(1))

# Null check
print("\nNulls per column:")
print(weather_enriched.isna().sum())

print(f"\nFinal shape: {weather_enriched.shape}")



--- MN ---
 year_month  avg_temp_f  heating_degree_days  cooling_degree_days
     202501        9.95               1662.0                  0.0
     202502       11.15               1466.0                  0.0
     202503       32.90                943.0                  0.0
     202504       42.20                655.0                  0.0
     202505       57.10                277.0                 28.0
     202506       65.20                 79.0                102.0
     202507       69.85                 19.0                202.0
     202508       67.15                 44.0                130.0
     202509       62.50                116.0                 57.0
     202510       51.15                421.0                  6.0
     202511       33.70                910.0                  0.0
     202512       13.40               1551.0                  0.0

--- FL ---
 year_month  avg_temp_f  heating_degree_days  cooling_degree_days
     202501       53.65                293.0        

### Step 4 — Overwrite the BigQuery Table With the Enriched Schema

We push the enriched DataFrame back into the same `aramark_spend.weather_state_monthly` table that we originally created. Using `write_disposition="WRITE_TRUNCATE"` tells BigQuery to drop the existing table and rebuild it from scratch with the new schema.

Because we overwrite rather than append, running this cell multiple times is safe, each run produces exactly the same 588-row table.


In [13]:
from google.cloud import bigquery

table_id = "big-data-algorithms-493312.aramark_spend.weather_state_monthly"

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"   # replace the 4-column table with the 8-column version
)

load_job = client.load_table_from_dataframe(
    weather_enriched,
    table_id,
    job_config=job_config,
)
load_job.result()

print(f"Upload complete: {table_id}")
print(f"Rows: {len(weather_enriched)}  |  Columns: {len(weather_enriched.columns)}")


Upload complete: big-data-algorithms-493312.aramark_spend.weather_state_monthly
Rows: 588  |  Columns: 8


### Step 5 — Verify the New Table in BigQuery

The upload cell confirms that the load job finished, but it's good practice to run a real query against the new table to make sure BigQuery sees what we expect. This query computes five quick sanity metrics in a single pass:

- `row_count` — should be 588 (49 states × 12 months)
- `state_count` — should be 49
- `month_count` — should be 12
- Average and maximum HDD and CDD across the entire table

The averages give us a national picture (how cold and hot was the country overall in 2025), and the maxima tell us how extreme the most intense month was.

In [14]:
# Confirm the table is back in BigQuery with the new schema.
verify_sql = """
SELECT
  COUNT(*)                       AS row_count,
  COUNT(DISTINCT state)          AS state_count,
  COUNT(DISTINCT year_month)     AS month_count,
  ROUND(AVG(heating_degree_days), 0) AS avg_hdd,
  ROUND(AVG(cooling_degree_days), 0) AS avg_cdd,
  ROUND(MAX(heating_degree_days), 0) AS max_hdd,
  ROUND(MAX(cooling_degree_days), 0) AS max_cdd
FROM `big-data-algorithms-493312.aramark_spend.weather_state_monthly`
"""
client.query(verify_sql).to_dataframe()


,row_count,state_count,month_count,avg_hdd,avg_cdd,max_hdd,max_cdd
0,588,49,12,414.0,102.0,1715.0,749.0


### Phase 4 complete

`weather_state_monthly` now contains 8 columns:

| Column | Type | Meaning |
|---|---|---|
| `state` | STRING | 2-letter state code |
| `year_month` | INT64 | YYYYMM integer |
| `avg_temp_f` | FLOAT | Monthly average temperature (°F) |
| `total_precip_inches` | FLOAT | Monthly total precipitation |
| `heating_degree_days` | FLOAT | Cold intensity — higher = colder |
| `cooling_degree_days` | FLOAT | Heat intensity — higher = hotter |
| `rain_threshold` | FLOAT | Per-state precipitation threshold used to derive the snow flag |
| `snow_proxy` | INT64 | Binary derived flag: `1` if month was cold *and* wet |

**Caveat on snow_proxy:** this is a *derived indicator*, not raw snowfall data. NOAA's standard statewide climate dataset does not include monthly snowfall. `heating_degree_days` is the primary cold-intensity measure we will use in Phase 5 correlations; `snow_proxy` remains as a complementary binary signal for the categorical hypothesis testing Tharun started.

**Coverage:** 49 US states. Hawaii, DC, and PR are not in the NOAA climdiv dataset. All Aramark spend in those locations is excluded from weather joins downstream.
